In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import dask.array as da
import os
from pycontrails.core import GeoVectorDataset
from pycontrails.models.gpat.gpat import GPAT, create_jobs_df, filter_jobs_df, load_fl_df, load_pl_df, load_chem_ds, plot_heatmap, anim_chem, mc_test, boxm_test

In [ ]:
outputs_dir = f"{os.getcwd()}/outputs/"

In [ ]:
# Filter criteria
criteria = {
        "n_ac": 0,
        "rt_fl": (pd.Timedelta(minutes=30), pd.Timedelta(hours=2)),
        "date_created": (pd.Timestamp("2024-11-15"), pd.Timestamp("2024-11-16"))
    }

In [ ]:
jobs_df = create_jobs_df(outputs_dir)

jobs_df

In [ ]:
filtered_df = filter_jobs_df(jobs_df, criteria)

job_ids = filtered_df.index.values


In [ ]:
fl_df = load_fl_df(job_ids, outputs_dir)

fl_df

plume_time_data = GeoVectorDataset(data=fl_df.loc[fl_df["time"] == fl_df["time"].min()])



In [ ]:
pl_df = load_pl_df(job_ids, outputs_dir)
pl_df.columns

pl_df

In [ ]:
chem_ds = load_chem_ds(job_ids, outputs_dir)

chem_ds = chem_ds.assign_coords(species_out=chem_ds.attrs["species_out"])

chem_ds_stacked = chem_ds.stack(
            {"cell": ["level", "longitude", "latitude"]}
        )
chem_ds_stacked = chem_ds_stacked.reset_index("cell")

max_emi_cell = chem_ds_stacked["emi"].mean(dim="time").argmax()#.item()
print(max_emi_cell)
# find cell that has max emissions averaged over time in it
cell_chem_ds = chem_ds.sel(job_id=job_ids[0], longitude=0.8, latitude=0.1).sel(level=178.6, method="nearest")
cell_chem_ds["emi"].sel(emi_species="CO")

# Select the emissions for the specified species
emi_data = cell_chem_ds["emi"].sel(emi_species="NO")
chem_data = cell_chem_ds["Y"].sel(species_out="NO")
#emi_data.plot()
chem_data.plot()

# # Convert time and emi data to pandas Series
# ts = 0
# time_series = pd.Series(emi_data["time"].values)
# emi_series = pd.Series(emi_data.values)

# # Print time and emi values side by side
# for time, emi in zip(time_series, emi_series):
#     ts += 1
#     print(f"TS: {ts}, Time: {time}, EMI: {emi}")

for s, species in enumerate(chem_ds["species"].values):
    print(chem_ds["bg_chem"].isel(level=1,latitude=0,longitude=0, species=s).values)


In [ ]:
#anim_chem(job_ids[0], jobs_df, fl_df, pl_df, chem_ds, var1="Y", var2="NO", level=178.6, resample_freq="2min")

In [ ]:
vecmass, gridmass, mc = mc_test(job_ids[0], jobs_df, fl_df, pl_df, chem_ds)
mc

In [ ]:
cell_chem_ds = boxm_test(job_ids[0], 0, chem_ds)

# dj_data = cell_chem_ds["DJ"].sel(photol_coeffs=3)
# dj_orig_data = cell_chem_ds["DJ_orig"].sel(photol_coeffs=3)
# dj_data.plot()
# dj_orig_data.plot()

chem_data = cell_chem_ds["Y"].sel(species_out="NO")
chem_orig_data = cell_chem_ds["Y_orig"].sel(species_out="NO")
chem_data.plot()
chem_orig_data.plot()
